# ELECTRE I-style outranking: comparing alternatives pairwise

This lab builds a crisp outranking relation from weighted concordance and range-normalised discordance. It demonstrates the relation-building stage only; it does not calculate an ELECTRE I kernel or a final recommendation. See the [method-family overview](../methods/electre-outranking.md) and [implementation contract](../implementations/python/electre-i/README.md).

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

current = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (current, *current.parents)
     if (candidate / "implementations" / "python" / "electre-i").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from a checkout of geo-dev-hub/spatial-analysis.')
package_path = repo_root / "implementations" / "python" / "electre-i"
sys.path.insert(0, str(package_path))
from electre_i import electre_i_relation


## 1. Define candidate alternatives

Imagine three candidate sites being assessed against access, community support and exposure cost. These values are synthetic. Higher access and support are preferred; lower exposure cost is preferred.

In [ ]:
performance = np.array([
    [8, 7, 4],  # Site A
    [6, 8, 5],  # Site B
    [5, 4, 7],  # Site C
], dtype=float)
names = ["Site A", "Site B", "Site C"]
criteria = ["Access", "Community support", "Exposure cost"]
weights = [0.5, 0.3, 0.2]
directions = ["benefit", "benefit", "cost"]

result = electre_i_relation(
    performance, weights, directions,
    concordance_threshold=0.6,
    discordance_threshold=0.3,
    alternative_names=names,
)
print("Normalised weights:", result.normalised_weights)
print("Outranking relation (rows outrank columns):\n", result.outranks)


## 2. Inspect concordance and discordance

Concordance measures the total weight of criteria supporting the claim that a is at least as good as b. Discordance measures the strongest relative disadvantage on any criterion where b performs better.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
matrices = [
    (result.concordance, "Concordance", 0, 1),
    (result.discordance, "Discordance", 0, 1),
    (result.outranks.astype(int), "Outranks (1 = yes)", 0, 1),
]
for ax, (matrix, title, low, high) in zip(axes, matrices):
    image = ax.imshow(matrix, cmap="viridis", vmin=low, vmax=high)
    ax.set_title(title)
    ax.set_xticks(range(len(names)), names, rotation=45, ha="right")
    ax.set_yticks(range(len(names)), names)
    ax.set_xlabel("Alternative being compared")
    ax.set_ylabel("Potential outranker")
    fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()


## 3. Read one pairwise comparison

The row is the potential outranker and the column is the alternative being compared. The diagonal is reflexive. Off-diagonal True values are the outranking assertions accepted by both thresholds.

In [ ]:
for i, a in enumerate(names):
    for j, b in enumerate(names):
        if i != j:
            print(f"{a} vs {b}: concordance={result.concordance[i, j]:.2f}, "
                  f"discordance={result.discordance[i, j]:.2f}, "
                  f"outranks={bool(result.outranks[i, j])}")


## 4. Explore threshold sensitivity

The thresholds are part of the decision model, not universal constants. Changing them can add or remove pairwise outranking assertions.

In [ ]:
for concordance_threshold in (0.5, 0.6, 0.7, 0.8):
    fitted = electre_i_relation(
        performance, weights, directions,
        concordance_threshold=concordance_threshold,
        discordance_threshold=0.3,
        alternative_names=names,
    )
    print(f"concordance threshold={concordance_threshold:.1f}")
    print(fitted.outranks.astype(int))


## 5. Questions to take away

1. Which criteria support Site A outranking Site B?
2. Which criterion creates the strongest discordance?
3. Why can two alternatives remain incomparable?
4. How do the thresholds change the relation?
5. Why is this not yet a complete ELECTRE I choice procedure?

Do not convert the relation into a total ranking without a documented exploitation procedure. The package does not calculate the ELECTRE I kernel.